# 01 · Validate GEO inputs

Download the six GSE96583 batch 2 supplements. Verify each archive against its recorded size and SHA-256. Match barcode columns to source metadata and validate sparse integer UMI counts. The verified archives and audit under `kang_ifnb` feed Stage 02.


## Runtime

Record the Python and package versions.


In [1]:
import sys
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("Kaggle working directory exists:", Path("/kaggle/working").is_dir())

for package in ["numpy", "pandas", "scipy", "anndata"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT_INSTALLED")

Python: 3.13.18
Working directory: /kaggle/working
Kaggle working directory exists: True
numpy: 2.1.3
pandas: 2.3.3
scipy: 1.16.3
anndata: NOT_INSTALLED


## Source access

Check that the GEO supplement endpoint is reachable.


In [2]:
from urllib.request import urlopen

url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE96nnn/GSE96583/suppl/GSE96583_batch2.genes.tsv.gz"
)

with urlopen(url, timeout=30) as response:
    print("NCBI HTTP status:", response.status)

NCBI HTTP status: 200


## Source manifest and downloads

Record the two accession-specific matrices, barcode lists and shared metadata, then verify the six archives.


In [3]:
import json

PROJECT = Path("/kaggle/working/kang_ifnb")
RAW_DIR = PROJECT / "data" / "raw"
AUDIT_DIR = PROJECT / "results" / "audit"

for folder in (RAW_DIR, AUDIT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SERIES = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE96nnn/GSE96583/suppl/"
CTRL = "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2560nnn/GSM2560248/suppl/"
STIM = "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2560nnn/GSM2560249/suppl/"

SOURCES = [
    {"url": SERIES + "GSE96583_batch2.total.tsne.df.tsv.gz",
     "bytes": 756342,
     "sha256": "1d57e72e92ca8695250e88cc0f1c3fa8c0be1175d974f8b427c58f1274dc6c09"},
    {"url": SERIES + "GSE96583_batch2.genes.tsv.gz",
     "bytes": 277054,
     "sha256": "93aa4e9b530ef9d6411ca129b416324c5cc1cc5a01a1fa6ed4f4a845480ed3ca"},
    {"url": CTRL + "GSM2560248_barcodes.tsv.gz",
     "bytes": 52904,
     "sha256": "d58d8d55cbe4a12757207784b3bc9227bf200c9100ca15131176e9f8159c955e"},
    {"url": STIM + "GSM2560249_barcodes.tsv.gz",
     "bytes": 52366,
     "sha256": "9bb38e080dfae81036fbcd9902c6c6254a4466aa85f70705479be3b2d6679d55"},
    {"url": CTRL + "GSM2560248_2.1.mtx.gz",
     "bytes": 28545918,
     "sha256": "32add28a0b3397d9ef3f220b7a6a55e98e60fe7b66fe48d0986d634df8ca0013"},
    {"url": STIM + "GSM2560249_2.2.mtx.gz",
     "bytes": 29050932,
     "sha256": "8aecc98a7ac4957bbc2570f87ebe8ce97332a5bcdbf557d40bd5aabfd287bdc5"},
]

reference = {"reference_verified_on": "2026-11-05", "files": SOURCES}
(AUDIT_DIR / "source_reference.json").write_text(
    json.dumps(reference, indent=2) + "\n", encoding="utf-8"
)

print("Project directory:", PROJECT)
print("Expected files:", len(SOURCES))
print("Expected total bytes:", sum(item["bytes"] for item in SOURCES))

Project directory: /kaggle/working/kang_ifnb
Expected files: 6
Expected total bytes: 58735516


In [4]:
import hashlib
import shutil
from datetime import datetime, timezone
from urllib.request import urlopen


def verify_file(path, expected_bytes, expected_sha256):
    observed_bytes = path.stat().st_size
    with path.open("rb") as stream:
        observed_sha256 = hashlib.file_digest(stream, "sha256").hexdigest()

    if observed_bytes != expected_bytes or observed_sha256 != expected_sha256:
        raise ValueError(
            f"Verification failed: {path.name}\n"
            f"Bytes: expected {expected_bytes}; observed {observed_bytes}\n"
            f"SHA-256: expected {expected_sha256}; observed {observed_sha256}"
        )

    return observed_bytes, observed_sha256


records = []

for source in SOURCES:
    filename = source["url"].rsplit("/", 1)[-1]
    destination = RAW_DIR / filename

    if destination.exists():
        size, digest = verify_file(
            destination, source["bytes"], source["sha256"]
        )
    else:
        temporary = RAW_DIR / (filename + ".part")
        print("Downloading:", filename, flush=True)

        try:
            with urlopen(source["url"], timeout=60) as response:
                with temporary.open("wb") as output:
                    shutil.copyfileobj(response, output, length=1024 * 1024)

            size, digest = verify_file(
                temporary, source["bytes"], source["sha256"]
            )
            temporary.replace(destination)

        except Exception:
            temporary.unlink(missing_ok=True)
            raise

    records.append({
        "filename": filename,
        "source_url": source["url"],
        "observed_bytes": size,
        "observed_sha256": digest,
        "size_match": size == source["bytes"],
        "sha256_match": digest == source["sha256"],
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    })

    print("Verified:", filename, flush=True)

Downloading: GSE96583_batch2.total.tsne.df.tsv.gz
Verified: GSE96583_batch2.total.tsne.df.tsv.gz
Downloading: GSE96583_batch2.genes.tsv.gz
Verified: GSE96583_batch2.genes.tsv.gz
Downloading: GSM2560248_barcodes.tsv.gz
Verified: GSM2560248_barcodes.tsv.gz
Downloading: GSM2560249_barcodes.tsv.gz
Verified: GSM2560249_barcodes.tsv.gz
Downloading: GSM2560248_2.1.mtx.gz
Verified: GSM2560248_2.1.mtx.gz
Downloading: GSM2560249_2.2.mtx.gz
Verified: GSM2560249_2.2.mtx.gz


## Validate the source structure

Read annotation and matrix headers, check dimensions and summarize donor coverage.


In [5]:
import pandas as pd

if len(records) != len(SOURCES):
    raise RuntimeError("The download cell has not verified all source files.")

manifest = pd.DataFrame(records)
manifest.to_csv(
    AUDIT_DIR / "download_manifest.tsv",
    sep="\t",
    index=False,
)

package_versions = {}

for package in ["numpy", "pandas", "scipy", "anndata"]:
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = None

runtime = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "working_directory": str(Path.cwd()),
    "package_versions": package_versions,
}

(AUDIT_DIR / "runtime_environment.json").write_text(
    json.dumps(runtime, indent=2) + "\n",
    encoding="utf-8",
)

display(
    manifest[["filename", "observed_bytes", "size_match", "sha256_match"]]
)

print("Verified files:", len(manifest))
print("Total bytes:", int(manifest["observed_bytes"].sum()))
print("Total MB (decimal):", manifest["observed_bytes"].sum() / 1_000_000)
print("Audit directory:", AUDIT_DIR)

,filename,observed_bytes,size_match,sha256_match
0,GSE96583_batch2.total.tsne.df.tsv.gz,756342,True,True
1,GSE96583_batch2.genes.tsv.gz,277054,True,True
2,GSM2560248_barcodes.tsv.gz,52904,True,True
3,GSM2560249_barcodes.tsv.gz,52366,True,True
4,GSM2560248_2.1.mtx.gz,28545918,True,True
5,GSM2560249_2.2.mtx.gz,29050932,True,True


Verified files: 6
Total bytes: 58735516
Total MB (decimal): 58.735516
Audit directory: /kaggle/working/kang_ifnb/results/audit


In [6]:
cell_meta_raw = pd.read_csv(
    RAW_DIR / "GSE96583_batch2.total.tsne.df.tsv.gz",
    sep="\t",
    index_col=0,
    dtype={
        "ind": "string", "stim": "string", "cluster": "string",
        "cell": "string", "multiplets": "string",
    },
)
cell_meta_raw.index.name = "metadata_cell_id"

genes_raw = pd.read_csv(
    RAW_DIR / "GSE96583_batch2.genes.tsv.gz",
    sep="\t",
    header=None,
    names=["gene_id", "gene_symbol"],
    dtype="string",
    keep_default_na=False,
)

barcodes_raw = {}
for accession in ["GSM2560248", "GSM2560249"]:
    barcodes_raw[accession] = pd.read_csv(
        RAW_DIR / f"{accession}_barcodes.tsv.gz",
        sep="\t", header=None, names=["barcode_original"],
        dtype="string", keep_default_na=False,
    )["barcode_original"]

print("Cell metadata shape:", cell_meta_raw.shape)
print("Gene metadata shape:", genes_raw.shape)
print("Cell metadata columns:", cell_meta_raw.columns.tolist())

display(cell_meta_raw.head(3))
display(genes_raw.head(3))

Cell metadata shape: (29065, 7)
Gene metadata shape: (35635, 2)
Cell metadata columns: ['tsne1', 'tsne2', 'ind', 'stim', 'cluster', 'cell', 'multiplets']


,tsne1,tsne2,ind,stim,cluster,cell,multiplets
metadata_cell_id,,,,,,,
AAACATACAATGCC-1,-4.277833,-19.294709,107,ctrl,5,CD4 T cells,doublet
AAACATACATTTCC-1,-27.640373,14.966629,1016,ctrl,9,CD14+ Monocytes,singlet
AAACATACCAGAAA-1,-27.493646,28.924885,1256,ctrl,9,CD14+ Monocytes,singlet


,gene_id,gene_symbol
0,ENSG00000243485,MIR1302-10
1,ENSG00000237613,FAM138A
2,ENSG00000186092,OR4F5


In [7]:
import gzip
from scipy.io import mminfo

matrix_sources = [
    ("GSM2560248", "ctrl", "GSM2560248_2.1.mtx.gz"),
    ("GSM2560249", "stim", "GSM2560249_2.2.mtx.gz"),
]

header_records = []

for accession, condition, filename in matrix_sources:
    with gzip.open(RAW_DIR / filename, "rb") as stream:
        n_rows, n_cols, entries, fmt, field, symmetry = mminfo(stream)

    n_metadata = int(cell_meta_raw["stim"].eq(condition).sum())

    if n_rows != len(genes_raw):
        raise ValueError(f"Gene-table length mismatch: {accession}")

    if n_cols != len(barcodes_raw[accession]) or n_cols != n_metadata:
        raise ValueError(f"Barcode/metadata length mismatch: {accession}")

    header_records.append({
        "source_sample_id": accession,
        "condition": condition,
        "n_gene_rows": n_rows,
        "n_barcode_columns": n_cols,
        "entries_declared": entries,
        "format": fmt,
        "field": field,
        "symmetry": symmetry,
    })

matrix_headers = pd.DataFrame(header_records)

if int(matrix_headers["n_barcode_columns"].sum()) != len(cell_meta_raw):
    raise ValueError("Metadata row total differs from the matrix column total.")

matrix_headers.to_csv(
    AUDIT_DIR / "matrix_headers.tsv",
    sep="\t",
    index=False,
)

display(matrix_headers)

,source_sample_id,condition,n_gene_rows,n_barcode_columns,entries_declared,format,field,symmetry
0,GSM2560248,ctrl,35635,14619,8732747,coordinate,real,general
1,GSM2560249,stim,35635,14446,8838098,coordinate,real,general


In [8]:
required_fields = {"ind", "stim", "cluster", "cell", "multiplets"}
missing_fields = required_fields - set(cell_meta_raw.columns)

if missing_fields:
    raise ValueError(
        f"Required metadata columns are absent: {sorted(missing_fields)}"
    )

metadata_missing = cell_meta_raw.isna().sum().rename("n_missing")
metadata_missing.rename_axis("source_field").to_csv(
    AUDIT_DIR / "metadata_missing.tsv",
    sep="\t",
)

shared_barcodes = (
    set(barcodes_raw["GSM2560248"])
    & set(barcodes_raw["GSM2560249"])
)

structure_checks = {
    "metadata_cell_ids_unique": bool(cell_meta_raw.index.is_unique),
    "gene_ids_unique": bool(genes_raw["gene_id"].is_unique),
    "empty_gene_ids": int(genes_raw["gene_id"].str.strip().eq("").sum()),
    "duplicate_gene_symbol_rows_after_first": int(
        genes_raw["gene_symbol"].duplicated().sum()
    ),
    "ctrl_barcodes_unique_within_sample": bool(
        barcodes_raw["GSM2560248"].is_unique
    ),
    "stim_barcodes_unique_within_sample": bool(
        barcodes_raw["GSM2560249"].is_unique
    ),
    "shared_barcode_strings_between_samples": len(shared_barcodes),
}

(AUDIT_DIR / "source_structure_checks.json").write_text(
    json.dumps(structure_checks, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(structure_checks, indent=2))
print("Missing metadata values:")
display(metadata_missing.to_frame())

{
  "metadata_cell_ids_unique": true,
  "gene_ids_unique": true,
  "empty_gene_ids": 0,
  "duplicate_gene_symbol_rows_after_first": 2697,
  "ctrl_barcodes_unique_within_sample": true,
  "stim_barcodes_unique_within_sample": true,
  "shared_barcode_strings_between_samples": 313
}
Missing metadata values:


,n_missing
tsne1,0
tsne2,0
ind,0
stim,0
cluster,0
cell,9
multiplets,0


In [9]:
if cell_meta_raw[["ind", "stim", "multiplets"]].isna().any().any():
    raise ValueError(
        "Missing donor, condition or demultiplexing label; review metadata."
    )

if set(cell_meta_raw["stim"]) != {"ctrl", "stim"}:
    raise ValueError("Unexpected condition labels; review the source.")

flag_counts = (
    cell_meta_raw.groupby(
        ["stim", "multiplets"],
        dropna=False,
        observed=True,
    )
    .size()
    .rename("n_barcodes")
    .reset_index()
)

flag_counts.to_csv(
    AUDIT_DIR / "demuxlet_labels_by_condition.tsv",
    sep="\t",
    index=False,
)

singlets_for_audit = cell_meta_raw.loc[
    cell_meta_raw["multiplets"].eq("singlet")
]

donor_condition_singlets = pd.crosstab(
    singlets_for_audit["ind"],
    singlets_for_audit["stim"],
).reindex(columns=["ctrl", "stim"], fill_value=0)

donor_condition_singlets.index.name = "donor_id"
donor_condition_singlets.columns.name = "condition"

donor_condition_singlets.to_csv(
    AUDIT_DIR / "donor_condition_singlets_before_rna_qc.tsv",
    sep="\t",
)

has_both_conditions = donor_condition_singlets.gt(0).all(axis=1)

print("Source demultiplexing labels:")
display(flag_counts)

print("Source singlets by donor and condition, before RNA QC:")
display(donor_condition_singlets)

print("Donor labels among source singlets:", len(donor_condition_singlets))
print(
    "Donors with >=1 source singlet in each condition:",
    int(has_both_conditions.sum()),
)
print("QC-eligible pairs for CD14+ analysis: not determined yet")

Source demultiplexing labels:


,stim,multiplets,n_barcodes
0,ctrl,ambs,706
1,ctrl,doublet,1598
2,ctrl,singlet,12315
3,stim,ambs,511
4,stim,doublet,1571
5,stim,singlet,12364


Source singlets by donor and condition, before RNA QC:


condition,ctrl,stim
donor_id,,
101,858,1166
1015,2738,2352
1016,1723,1635
1039,461,647
107,517,525
1244,1879,1464
1256,2108,2026
1488,2031,2549


Donor labels among source singlets: 8
Donors with >=1 source singlet in each condition: 8
QC-eligible pairs for CD14+ analysis: not determined yet


## Align barcodes to metadata

Use source sample and barcode to identify cells. Verify the metadata aliases for barcode strings shared across captures.


In [10]:
mapping_parts = []
alignment_records = []

for accession, condition, _ in matrix_sources:
    original = barcodes_raw[accession].tolist()
    source_ids = cell_meta_raw.loc[
        cell_meta_raw["stim"].eq(condition)
    ].index.tolist()

    expected_ids = [
        barcode + "1"
        if condition == "stim" and barcode in shared_barcodes
        else barcode
        for barcode in original
    ]

    positions_match = source_ids == expected_ids

    if not positions_match:
        raise ValueError(
            f"Unverified barcode-to-metadata order: {accession}. Stop here."
        )

    part = pd.DataFrame({
        "cell_id": [accession + "__" + barcode for barcode in original],
        "source_sample_id": accession,
        "condition": condition,
        "matrix_column_zero_based": range(len(original)),
        "barcode_original": original,
        "metadata_cell_id": source_ids,
    })

    n_aliases = int(
        part["barcode_original"].ne(part["metadata_cell_id"]).sum()
    )

    mapping_parts.append(part)
    alignment_records.append({
        "source_sample_id": accession,
        "n_barcodes": len(original),
        "n_source_id_aliases": n_aliases,
        "all_positions_verified": positions_match,
    })

cell_mapping = pd.concat(mapping_parts, ignore_index=True)

if cell_mapping["metadata_cell_id"].tolist() != cell_meta_raw.index.tolist():
    raise ValueError(
        "Combined metadata order differs from CTRL-then-STIM source order."
    )

if (
    not cell_mapping["cell_id"].is_unique
    or not cell_mapping["metadata_cell_id"].is_unique
):
    raise ValueError("The cell mapping is not one-to-one.")

alignment_checks = pd.DataFrame(alignment_records)

cell_mapping.to_csv(
    AUDIT_DIR / "cell_mapping.tsv.gz",
    sep="\t",
    index=False,
)

alignment_checks.to_csv(
    AUDIT_DIR / "barcode_alignment_checks.tsv",
    sep="\t",
    index=False,
)

display(alignment_checks)
print("Mapped barcode columns:", len(cell_mapping))
print("Source-ID alias examples:")

display(
    cell_mapping.loc[
        cell_mapping["barcode_original"].ne(cell_mapping["metadata_cell_id"]),
        ["source_sample_id", "barcode_original", "metadata_cell_id"],
    ].head(3)
)

,source_sample_id,n_barcodes,n_source_id_aliases,all_positions_verified
0,GSM2560248,14619,0,True
1,GSM2560249,14446,313,True


Mapped barcode columns: 29065
Source-ID alias examples:


,source_sample_id,barcode_original,metadata_cell_id
14659,GSM2560249,AAACGCTGTGTCAG-1,AAACGCTGTGTCAG-11
14664,GSM2560249,AAACGGCTTACTCT-1,AAACGGCTTACTCT-11
14669,GSM2560249,AAACTTGACGTGTA-1,AAACTTGACGTGTA-11


## Validate and summarize counts

Check sparse coordinates, dimensions, integer UMI counts and the total across both captures.


In [11]:
import numpy as np
from scipy.io import mmread
from scipy.sparse import issparse

counts_by_sample = {}
count_records = []

for accession, condition, filename in matrix_sources:
    print("Reading counts:", accession, flush=True)

    with gzip.open(RAW_DIR / filename, "rb") as stream:
        source_counts = mmread(stream, spmatrix=True)

    expected_shape = (
        len(genes_raw),
        len(barcodes_raw[accession]),
    )

    if (
        not issparse(source_counts)
        or source_counts.shape != expected_shape
    ):
        raise ValueError(
            f"Unexpected count matrix type or dimensions: {accession}"
        )

    values = source_counts.data

    checks = {
        "finite": bool(np.isfinite(values).all()),
        "nonnegative": bool((values >= 0).all()),
        "integer_valued": bool((values == np.floor(values)).all()),
    }

    if not all(checks.values()):
        raise ValueError(
            f"Invalid count values in {accession}: {checks}"
        )

    if values.size and values.max() >= 2**63:
        raise ValueError(
            f"A count is outside the supported int64 range: {accession}"
        )

    counts = source_counts.astype(np.int64).tocsr()

    duplicate_entries = int(source_counts.nnz - counts.nnz)

    if duplicate_entries != 0:
        raise ValueError(
            f"Repeated matrix coordinates require review: {accession}"
        )

    explicit_zeros = int((counts.data == 0).sum())
    counts.eliminate_zeros()

    counts_by_sample[accession] = counts

    count_records.append({
        "source_sample_id": accession,
        "condition": condition,
        "n_genes": counts.shape[0],
        "n_barcodes": counts.shape[1],
        "source_dtype": str(source_counts.dtype),
        "validated_dtype": str(counts.dtype),
        **checks,
        "duplicate_coordinates": duplicate_entries,
        "explicit_zero_entries": explicit_zeros,
        "nnz": int(counts.nnz),
        "total_umi_all_source_barcodes": int(counts.sum()),
    })

    del source_counts, values

counts_checks = pd.DataFrame(count_records)

counts_checks.to_csv(
    AUDIT_DIR / "raw_count_checks.tsv",
    sep="\t",
    index=False,
)

display(counts_checks)

Reading counts: GSM2560248
Reading counts: GSM2560249


,source_sample_id,condition,n_genes,n_barcodes,source_dtype,validated_dtype,finite,nonnegative,integer_valued,duplicate_coordinates,explicit_zero_entries,nnz,total_umi_all_source_barcodes
0,GSM2560248,ctrl,35635,14619,float64,int64,True,True,True,0,0,8732747,23540929
1,GSM2560249,stim,35635,14446,float64,int64,True,True,True,0,0,8838098,25059996


In [12]:
if len(counts_by_sample) != len(matrix_sources):
    raise RuntimeError(
        "Count validation did not finish for all source samples."
    )

counts_runtime = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": version("scipy"),
}

(AUDIT_DIR / "counts_runtime.json").write_text(
    json.dumps(counts_runtime, indent=2) + "\n",
    encoding="utf-8",
)

print("Validated sparse matrices:", len(counts_by_sample))
print("Mapped barcode columns:", len(cell_mapping))
print("Unique gene IDs:", genes_raw["gene_id"].nunique())

print(
    "Total UMI, all source barcodes:",
    int(counts_checks["total_umi_all_source_barcodes"].sum()),
)

print("QC-eligible CD14+ pairs: not determined yet")

print(
    "Runtime:",
    {
        key: counts_runtime[key]
        for key in ["numpy", "pandas", "scipy"]
    },
)

Validated sparse matrices: 2
Mapped barcode columns: 29065
Unique gene IDs: 35635
Total UMI, all source barcodes: 48600925
QC-eligible CD14+ pairs: not determined yet
Runtime: {'numpy': '2.1.3', 'pandas': '2.3.3', 'scipy': '1.16.3'}
